In [1]:
import numpy as np
import os
import pandas as pd
from matplotlib import pyplot as plt

#---%% parameters for this study test ---
number_real=3000
np.random.seed = 202
v_shale = np.round(np.random.uniform(0.01, 0.99, number_real), 3)

angles_alpha= np.random.randint(0, 360, number_real)
angles_beta= np.random.randint(0, 360, number_real)
angles_gamma= np.random.randint(0, 360, number_real)
h_x = np.round(np.concatenate((np.random.uniform(0.7, 3, int(number_real*0.7)),
                               np.random.uniform(3, 5, number_real - int(number_real*0.3)))), 3)
h_y = np.round(np.concatenate((np.random.uniform(0.7, 3, int(number_real*0.7)),
                               np.random.uniform(3, 5, number_real - int(number_real*0.3)))), 3)
h_z = np.round(np.concatenate((np.random.uniform(0.1, 0.5, int(number_real*0.8)),
                               np.random.uniform(0.6, 1.0, number_real - int(number_real*0.2)))), 3)

if os.path.exists('sis_parameters_20241002.csv' ):
    os.remove('sis_parameters_20241002.csv' )
sis_parameters = []
for i, _ in enumerate(range(number_real)):
    sis_parameters.append([i, h_x[i], h_y[i], h_z[i], angles_alpha[i], angles_beta[i], angles_gamma[i], v_shale[i]])
df = pd.DataFrame(sis_parameters)#, columns=['Test Number']
file_path = 'sis_parameters_20241002.csv'  # Replace with your desired output file path
df.to_csv(file_path, index=False,header=None)
print(f"sis_parameters test numbers saved to {file_path}")

sis_parameters test numbers saved to sis_parameters_20241002.csv


In [7]:
num_i = 20;        # i_direction, y grid
num_j = 20;        # j_direction, x grid
num_k = 20;        # k_direction, z grid
shale_proportion_threshold = 5/100 #5/100
num_realization = 30;


shale_volums = []

if os.path.exists("sisim_20241002.out"):
    os.remove("sisim_20241002.out")
if os.path.exists("SIS_20241002.par"):
    os.remove("SIS_20241002.par")    


for idx_run in range(0, len(sis_parameters)):
    correlation_length_x = sis_parameters[idx_run][1]
    correlation_length_y = sis_parameters[idx_run][2]
    correlation_length_z = sis_parameters[idx_run][3]
    correlation_anglealpha = sis_parameters[idx_run][4]
    correlation_anglebeta = sis_parameters[idx_run][5]
    correlation_anglegamma = sis_parameters[idx_run][6]
    shale_proportion = round(sis_parameters[idx_run][7], 3);
    sand_proportion = 1 - shale_proportion;


    #%% generate parameter input file for SIS
    with open('SIS_20241002.par', 'w') as fid:
        fid.write('************************************************** \n')
        fid.write('    Parameters for SIS \n')
        fid.write('************************************************** \n\n')
        fid.write('START OF PARAMETERS: \n'); 
        fid.write('0                               -   1 =continuous(cdf), 0 =categorical(pdf) \n');
        fid.write('2                               -   number thresholds/categories \n');
        fid.write('0  1                            -   thresholds / categories \n');
        fid.write(str(sand_proportion) + '  ' + str(shale_proportion) + ' -   global cdf / pdf \n');
        fid.write('none.dat                        -   file with input \n');
        fid.write('1   2   3   4   5               -  columns for X,Y,Z, and variable \n');#2  3  4  5  6  7 #1  2  3  4  5  6 
        fid.write('direcik                         -   file with soft indicator input \n');
        fid.write('1   2   3   4   5               -    columns for X,Y,Z, and indicators \n');
        fid.write('0                               -   Markov-Bayes simulation (0=no,1=yes) \n');
        fid.write('0.61  0.54  0.56  0.53  0.29    -   calibration B(z) values; no needed if 0 is markov-bayes \n');
        fid.write('-1.0e21    1.0e21               -   trimming limits \n');
        fid.write('0.0   1.0                       -   minimum and maximum data value \n');
        fid.write('1     0.0                       -   lower tail option and parameter \n');
        fid.write('1     1                         -   middle tail option and parameter \n');
        fid.write('1     1                         -   upper tail option and parameter \n');
        fid.write('none.dat                        -   file with tabulated values \n');
        fid.write('3  0                            -   columns for variable, weight \n');#5 6 7  0/ 3 0
        fid.write('3                               -   debugging level: 0,1,2,3 \n');
        fid.write('sisim_20241002.dbg                  -   file for debugging output \n');
        fid.write('sisim_20241002.out                  -   file for simulation output \n');
        
        fid.write(str(num_realization) +         ' -   number of realizations \n');
        fid.write(str(num_i) +  '  0.025   0.05       -   nx,xmn,xsize \n');
        fid.write(str(num_j) +  '  0.025   0.05       -   ny,ymn,ysize \n');
        fid.write(str(num_k) +  '  0.025   0.05        -   nz,zmn,zsiz \n');
        
        fid.write('69069                           -   random number seed \n'); #123456/69069
        fid.write('12                              -   maximum original data for each kriging \n');
        fid.write('12                              -   maximum previous nodes for each kriging \n');
        fid.write('1                               -   maximum soft indicator nodes for kriging \n');
        fid.write('1                               -   assign data to nodes? (0=no,1=yes) \n');
        fid.write('0   3                           -   multiple grid search? (0=no,1=yes),num \n');
        fid.write('0                               -   maximum per octant    (0=not used) \n');
        fid.write('20.0 20.0  20.0                 -   maximum search radii \n');
        fid.write('0.0  0.0  0.0                   -   angles for search ellipsoid \n');
        fid.write('21   21    11                   -   size of covariance lookup table \n');
        fid.write('0   1                           -   0=full IK, 1=median approx. (cutoff) \n');#2.5 #1
        fid.write('0                               -   0=SK, 1=OK \n');
        fid.write('1    0.15                       -   One   nst, nugget effect \n');
        fid.write('1    0.85 '  + str(correlation_anglealpha) 
                  + '  ' + str(correlation_anglebeta)
                  + '  ' + str(correlation_anglegamma) 
                  + '                              -   it, cc, ang1, ang2, ang3 \n');
        fid.write('     '  + str(correlation_length_x) 
                  + '   ' +  str(correlation_length_y) 
                  + '   ' +  str(correlation_length_z) 
                  + '                              -   a_hmax, a_hmin, a_vert \n');
        fid.write('1    0.10                       -   Two   nst, nugget effect \n');
        fid.write('1    0.90 ' + str(correlation_anglealpha) 
                  + '  ' + str(correlation_anglebeta)
                  + '  ' + str(correlation_anglegamma) 
                  + '                              -   it, cc, ang1, ang2, ang3 \n');
        fid.write('     '  + str(correlation_length_x) 
                  + '   ' +  str(correlation_length_y) 
                  + '   ' +  str(correlation_length_z) 
                  + '                              -   a_hmax, a_hmin, a_vert \n\n');
        fid.close() 

        #%% run sequential indicator simulation exe in GsLib
        os.system("sisim.exe SIS_20241002.par")

        #%% screen out non-realistic reali
        multi_realizations_vector = np.genfromtxt('sisim_20241002.out', skip_header=3);

        multi_realizations_shale_volume = []
        for idx_real in range(0, num_realization):
            cur_real_vector = multi_realizations_vector[idx_real * num_i * num_j * num_k :(idx_real+1) * num_i * num_j * num_k]
            cur_shale_volume = np.sum(cur_real_vector) / (num_i * num_j * num_k)
            multi_realizations_shale_volume.append(cur_shale_volume)
        multi_realizations_shale_volume = np.array(multi_realizations_shale_volume)

        # calculate the relative error between each realization and predefined shale volume
        relative_error = abs(multi_realizations_shale_volume - shale_proportion) / shale_proportion
        best_realization_index = np.argmin(relative_error)
        best_realization_relative_error = relative_error[best_realization_index]
        best_realization_vector = multi_realizations_vector[best_realization_index * num_i * num_j * num_k:(best_realization_index+1) * num_i * num_j * num_k]
        actual_shale_volume = multi_realizations_shale_volume[best_realization_index]
        shale_volums.append(actual_shale_volume)

        #% save best realization to dat file
        best_realization_matrix_transposted = np.transpose(np.reshape(best_realization_vector, (num_k, num_j, num_i)))
        best_realization_vector_transposted = np.reshape(best_realization_matrix_transposted, (num_k*num_j*num_i,))
        if not os.path.exists("realizations_20241002"):
            os.makedirs("realizations_20241002")
        fid = open('realizations_20241002/realization_' + str(idx_run) + '.dat', 'w')
        for idx in range(0, num_i * num_j * num_k):
            #fid.write(str(int(idx + 1)) + '\t')
            fid.write(str(int(best_realization_vector_transposted[idx])) + '\n')
        fid.close()

        #%% save parameters ------------------------------------------------------------
fid = open('actual_shale_volumes_20241002.dat', 'w')
for idx in range(0, len(shale_volums)):
    fid.write(str(shale_volums[idx]) + '\n')
fid.close()
